In [1]:
import pandas_datareader.data as web #to collect data
import datetime as dt #to specify start and end dates

# import yfinance as yf

import eventstudy as es
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns


import pandas as pd

import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.regression.rolling import RollingOLS

from patsy import dmatrices
from tqdm.notebook import tqdm
tqdm.pandas()

## Data reading and melting

In [2]:
import_folder_path = r"..\..\[IN USE] Rookie Directors\[1] Director Level\director_wrangle_output"
output_folder_path = "car_output1"
supporting_folder_path = "supporting_datafiles"

In [3]:
data = pd.read_csv(rf"{supporting_folder_path}\Adjusted Clos_collated.csv").drop("Unnamed: 0", axis = 1)

In [4]:
data

,AsOnDate,20 Microns Ltd.,20Th Century Finance Corpn. Ltd. [Merged],360 One Wam Ltd.,3I Infotech Ltd.,3M India Ltd.,3P Land Holdings Ltd.,3Rd Rock Multimedia Ltd.,5Paisa Capital Ltd.,63 Moons Technologies Ltd.,...,3898,3899,3900,3901,3902,3903,3904,3905,3906,3907
0,2004-01-01,NaN,NaN,NaN,NaN,515.00,NaN,NaN,NaN,NaN,...,66.83,NaN,15.25,NaN,NaN,NaN,59.85,25.79,NaN,NaN
1,2004-01-02,NaN,NaN,NaN,NaN,530.50,3.20,NaN,NaN,NaN,...,65.88,NaN,17.35,NaN,NaN,NaN,66.50,26.74,NaN,NaN
2,2004-01-05,NaN,NaN,NaN,NaN,511.75,3.42,NaN,NaN,NaN,...,61.76,NaN,16.30,NaN,NaN,NaN,62.70,25.95,NaN,NaN
3,2004-01-06,NaN,NaN,NaN,NaN,495.00,NaN,NaN,NaN,NaN,...,61.94,NaN,15.70,NaN,NaN,NaN,59.50,25.74,NaN,NaN
4,2004-01-07,NaN,NaN,NaN,NaN,490.40,2.76,NaN,NaN,NaN,...,60.33,NaN,16.00,NaN,NaN,NaN,57.75,25.15,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5025,2024-03-21,144.70,12.5,673.40,41.60,30049.25,30.80,64.6,487.45,407.15,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5026,2024-03-22,145.05,12.5,665.85,41.85,30728.40,30.40,64.6,487.85,402.95,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5027,2024-03-26,143.00,12.5,650.30,39.20,30487.15,28.90,64.6,476.05,393.75,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5028,2024-03-27,143.00,12.5,667.95,39.40,31475.10,28.40,64.6,490.30,381.60,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
dataLong = data.melt( id_vars = "AsOnDate", value_vars = data.columns[1:3908]).rename({"value":"ACP", "variable":"CompanyName"}, axis = 1).drop_duplicates().reset_index(drop = True)
dataLong = dataLong.loc[~( (dataLong.duplicated(subset = ["CompanyName", "AsOnDate"], keep = False)) & (dataLong["ACP"].isnull())) ]
dataLong = dataLong.loc[~ dataLong.duplicated(subset = ["CompanyName", "AsOnDate"], keep = False)].drop_duplicates().reset_index(drop = True)

In [6]:
dataLong

,AsOnDate,CompanyName,ACP
0,2004-01-01,20 Microns Ltd.,NaN
1,2004-01-02,20 Microns Ltd.,NaN
2,2004-01-05,20 Microns Ltd.,NaN
3,2004-01-06,20 Microns Ltd.,NaN
4,2004-01-07,20 Microns Ltd.,NaN
...,...,...,...
19640110,2024-03-21,3907,NaN
19640111,2024-03-22,3907,NaN
19640112,2024-03-26,3907,NaN
19640113,2024-03-27,3907,NaN


## Data Cleaning

### Data Snipping from either ends

In [7]:
def dataSnip(frame):
    
    first_valid_idx = frame["ACP"].first_valid_index()
    
    if first_valid_idx is not None:
        frame = frame.loc[first_valid_idx:]
        
    else:
        frame = frame.iloc[0:0]

    last_valid_idx = frame["ACP"].last_valid_index()
    
    if last_valid_idx is not None:
        frame = frame.loc[:last_valid_idx]
        
    else:
        frame = frame
        
    return frame

In [8]:
dataLong2 = dataLong.groupby(by="CompanyName").progress_apply(dataSnip).reset_index(drop=True)
dataLong2

  0%|          | 0/3907 [00:00<?, ?it/s]

C:\Users\SHIVAM\anaconda3\Lib\site-packages\tqdm\std.py:805: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return getattr(df, df_function)(wrapper, **kwargs)


,AsOnDate,CompanyName,ACP
0,2008-10-06,20 Microns Ltd.,16.82
1,2008-10-07,20 Microns Ltd.,15.05
2,2008-10-08,20 Microns Ltd.,13.25
3,2008-10-10,20 Microns Ltd.,11.60
4,2008-10-13,20 Microns Ltd.,12.32
...,...,...,...
13465088,2024-03-21,Zylog Systems Ltd.,0.35
13465089,2024-03-22,Zylog Systems Ltd.,0.35
13465090,2024-03-26,Zylog Systems Ltd.,0.35
13465091,2024-03-27,Zylog Systems Ltd.,0.35


### Inter series NaN values filled as the previous value

In [9]:
def dataForwardFill(frame):

    frame["ACP"] = frame["ACP"].ffill()
        
    return frame

In [10]:
dataLong3 = dataLong2.groupby(by="CompanyName").apply(dataForwardFill).reset_index(drop=True)
dataLong3

C:\Users\SHIVAM\AppData\Local\Temp\ipykernel_15268\3844389372.py:1: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  dataLong3 = dataLong2.groupby(by="CompanyName").apply(dataForwardFill).reset_index(drop=True)


,AsOnDate,CompanyName,ACP
0,2008-10-06,20 Microns Ltd.,16.82
1,2008-10-07,20 Microns Ltd.,15.05
2,2008-10-08,20 Microns Ltd.,13.25
3,2008-10-10,20 Microns Ltd.,11.60
4,2008-10-13,20 Microns Ltd.,12.32
...,...,...,...
13465088,2024-03-21,Zylog Systems Ltd.,0.35
13465089,2024-03-22,Zylog Systems Ltd.,0.35
13465090,2024-03-26,Zylog Systems Ltd.,0.35
13465091,2024-03-27,Zylog Systems Ltd.,0.35


### Result

In [11]:
dataLong3.ACP.isnull().value_counts()

ACP
False    13465093
Name: count, dtype: int64

In [12]:
closingPrice = dataLong3.copy()

## Simple Returns

In [13]:
def pct_change(frame):
    frame = frame.sort_values(by = ["AsOnDate"])
    frame["pct"] = frame["ACP"].pct_change(fill_method = None)
    return frame

In [14]:
simpleReturn = closingPrice.groupby("CompanyName").apply(pct_change).reset_index(drop = True)

C:\Users\SHIVAM\AppData\Local\Temp\ipykernel_15268\3133524175.py:1: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  simpleReturn = closingPrice.groupby("CompanyName").apply(pct_change).reset_index(drop = True)


In [15]:
simpleReturn["AsOnDate"] = pd.to_datetime(simpleReturn["AsOnDate"], format = "%Y-%m-%d")
simpleReturn = simpleReturn.drop_duplicates().reset_index(drop = True)
simpleReturn

,AsOnDate,CompanyName,ACP,pct
0,2008-10-06,20 Microns Ltd.,16.82,NaN
1,2008-10-07,20 Microns Ltd.,15.05,-0.105232
2,2008-10-08,20 Microns Ltd.,13.25,-0.119601
3,2008-10-10,20 Microns Ltd.,11.60,-0.124528
4,2008-10-13,20 Microns Ltd.,12.32,0.062069
...,...,...,...,...
13465088,2024-03-21,Zylog Systems Ltd.,0.35,0.000000
13465089,2024-03-22,Zylog Systems Ltd.,0.35,0.000000
13465090,2024-03-26,Zylog Systems Ltd.,0.35,0.000000
13465091,2024-03-27,Zylog Systems Ltd.,0.35,0.000000


## FF 3 constants

In [16]:
ff3Const = pd.read_csv(rf"{supporting_folder_path}\2024-03_FourFactors_and_Market_Returns_Daily_SurvivorshipBiasAdjusted.csv")

In [17]:
ff3Const["date"] = pd.to_datetime(ff3Const["date"], format = "%d-%m-%Y")

In [18]:
ff3Const = ff3Const.rename({"date":"AsOnDate"}, axis = 1).drop_duplicates().reset_index(drop = True)
ff3Const["RMRF"] = ff3Const["MF"] - ff3Const["RF"]
ff3Const.columns[[0,1,2,3,4,5,6]]
orderedCol = ff3Const.columns[[0, 5, 6, 4, 1, 2]]
ff3ConstOrdered = ff3Const[orderedCol]
ff3ConstOrdered

,AsOnDate,RF,RMRF,MF,SMB,HML
0,1993-10-01,NaN,NaN,NaN,1.414154,1.182552
1,1993-10-04,0.022014,-0.954330,-0.932316,0.472301,0.371959
2,1993-10-05,0.022014,-0.315512,-0.293498,0.046619,0.839734
3,1993-10-06,0.022014,-0.352836,-0.330823,-0.085561,-1.492747
4,1993-10-07,0.022014,0.360946,0.382959,-0.286668,0.135259
...,...,...,...,...,...,...
7563,2024-03-21,0.018215,1.441807,1.460021,0.572866,1.270303
7564,2024-03-22,0.018215,0.593324,0.611538,0.752002,0.124527
7565,2024-03-26,0.072878,-0.044514,0.028364,-1.091597,0.440605
7566,2024-03-27,0.018215,0.326702,0.344916,-0.165072,-0.188505


In [19]:
# df = ff3ConstOrdered.copy()
# df["Year"] = float(df["AsOnDate"].year)
# df.groupby(["Year"]).apply(

## Merged and Final Data set

In [20]:
dataMerged1 = simpleReturn.merge(ff3ConstOrdered, on = "AsOnDate", how = "inner").sort_values(["CompanyName", "AsOnDate"]).set_index(["CompanyName", "AsOnDate"]).reset_index().drop_duplicates().reset_index(drop = True)

In [21]:
dataMerged1

,CompanyName,AsOnDate,ACP,pct,RF,RMRF,MF,SMB,HML
0,20 Microns Ltd.,2008-10-06,16.82,NaN,0.069713,-6.381449,-6.311735,-0.373052,-0.566450
1,20 Microns Ltd.,2008-10-07,15.05,-0.105232,0.023232,-0.669144,-0.645911,-1.502487,0.184699
2,20 Microns Ltd.,2008-10-08,13.25,-0.119601,0.023232,-3.533362,-3.510130,-1.780674,0.072932
3,20 Microns Ltd.,2008-10-10,11.60,-0.124528,0.045520,-7.052324,-7.006804,0.217126,0.354629
4,20 Microns Ltd.,2008-10-13,12.32,0.062069,0.066863,5.042738,5.109602,-2.437097,0.145853
...,...,...,...,...,...,...,...,...,...
13465088,Zylog Systems Ltd.,2024-03-21,0.35,0.000000,0.018215,1.441807,1.460021,0.572866,1.270303
13465089,Zylog Systems Ltd.,2024-03-22,0.35,0.000000,0.018215,0.593324,0.611538,0.752002,0.124527
13465090,Zylog Systems Ltd.,2024-03-26,0.35,0.000000,0.072878,-0.044514,0.028364,-1.091597,0.440605
13465091,Zylog Systems Ltd.,2024-03-27,0.35,0.000000,0.018215,0.326702,0.344916,-0.165072,-0.188505


## Company Keys and Merging with data above

In [22]:
companyKeys = pd.read_excel(rf"{supporting_folder_path}\Prowess Code_NSE Symbol.xlsx").rename({"Company Name":"CompanyName", "Prowess company code":"ProwessCode", "NSE symbol":"Symbol"}, axis = 1 )

In [23]:
companyKeys

,CompanyName,ProwessCode,Symbol
0,'K' Steamship Agencies Pvt. Ltd.,3,NaN
1,'X'Clusive Business Centre Pvt. Ltd.,307865,NaN
2,1 To 1 Help.Net Pvt. Ltd.,591675,NaN
3,10C India Internet Pvt. Ltd.,556976,NaN
4,10I Commerce Services Pvt. Ltd.,560502,NaN
...,...,...,...
55307,Zylog Systems Ltd.,275793,ZYLOG
55308,Zyma Laboratories Ltd. [Merged],275794,NaN
55309,Zyphar'S Pharmaceutics Pvt. Ltd.,565342,NaN
55310,Zytel Agencies Ltd.,275795,NaN


In [24]:
dataMerged2 = companyKeys.merge(dataMerged1, on = "CompanyName", how = "right").sort_values( by = ["CompanyName", "AsOnDate"])

In [25]:
# The above are not in the director dataset, so can be ignored
dataMerged2

,CompanyName,ProwessCode,Symbol,AsOnDate,ACP,pct,RF,RMRF,MF,SMB,HML
0,20 Microns Ltd.,11.0,20MICRONS,2008-10-06,16.82,NaN,0.069713,-6.381449,-6.311735,-0.373052,-0.566450
1,20 Microns Ltd.,11.0,20MICRONS,2008-10-07,15.05,-0.105232,0.023232,-0.669144,-0.645911,-1.502487,0.184699
2,20 Microns Ltd.,11.0,20MICRONS,2008-10-08,13.25,-0.119601,0.023232,-3.533362,-3.510130,-1.780674,0.072932
3,20 Microns Ltd.,11.0,20MICRONS,2008-10-10,11.60,-0.124528,0.045520,-7.052324,-7.006804,0.217126,0.354629
4,20 Microns Ltd.,11.0,20MICRONS,2008-10-13,12.32,0.062069,0.066863,5.042738,5.109602,-2.437097,0.145853
...,...,...,...,...,...,...,...,...,...,...,...
13465088,Zylog Systems Ltd.,275793.0,ZYLOG,2024-03-21,0.35,0.000000,0.018215,1.441807,1.460021,0.572866,1.270303
13465089,Zylog Systems Ltd.,275793.0,ZYLOG,2024-03-22,0.35,0.000000,0.018215,0.593324,0.611538,0.752002,0.124527
13465090,Zylog Systems Ltd.,275793.0,ZYLOG,2024-03-26,0.35,0.000000,0.072878,-0.044514,0.028364,-1.091597,0.440605
13465091,Zylog Systems Ltd.,275793.0,ZYLOG,2024-03-27,0.35,0.000000,0.018215,0.326702,0.344916,-0.165072,-0.188505


In [26]:
dataMerged2["AsOnDate"] = pd.to_datetime(dataMerged2["AsOnDate"], format = "%Y-%m-%d")

## Director Dates of study

In [27]:
dir = pd.read_pickle(rf"{import_folder_path}\Main_Director_COMPLETE.pkl")

In [28]:
dirAppointment = dir.loc[:, ["Symbol", "Appointment Date"]].copy().sort_values( by = ["Symbol", "Appointment Date"]).drop_duplicates().dropna().reset_index(drop = True).rename({"Appointment Date":"Date of Study"}, axis = 1)
dirAppointment

,Symbol,Date of Study
0,20MICRONS,1988-03-29
1,20MICRONS,1998-07-02
2,20MICRONS,2000-04-10
3,20MICRONS,2001-01-29
4,20MICRONS,2001-02-27
...,...,...
36768,ZYLOG,2014-11-19
36769,ZYLOG,2015-08-14
36770,ZYLOG,2015-11-25
36771,ZYLOG,2016-11-23


In [29]:
dirCessation = dir.loc[:, ["Symbol", "Cessation Date"]].copy().sort_values( by = ["Symbol", "Cessation Date"]).drop_duplicates().dropna().reset_index(drop = True).rename({"Cessation Date":"Date of Study"}, axis = 1)
dirCessation

,Symbol,Date of Study
0,20MICRONS,2009-03-02
1,20MICRONS,2009-07-29
2,20MICRONS,2009-07-30
3,20MICRONS,2011-04-29
4,20MICRONS,2011-10-22
...,...,...
25802,ZYLOG,2014-11-19
25803,ZYLOG,2015-11-14
25804,ZYLOG,2016-06-30
25805,ZYLOG,2016-08-12


In [30]:
dirDemise = dir.loc[:, ["Symbol", "Date of Demise"]].copy().sort_values( by = ["Symbol", "Date of Demise"]).drop_duplicates().dropna().reset_index(drop = True).rename({"Date of Demise":"Date of Study"}, axis = 1)
dirDemise

,Symbol,Date of Study
0,20MICRONS,2009-07-29
1,20MICRONS,2021-06-09
2,3PLAND,2009-06-14
3,3PLAND,2014-12-23
4,AARTIDRUGS,2021-07-16
...,...,...
1217,ZODJRDMKJ,2010-01-18
1218,ZUARI,2022-06-12
1219,ZUARIIND,2008-08-30
1220,ZUARIIND,2016-05-30


### All dates combined

In [31]:
dirDates = pd.concat([
    dirAppointment, dirCessation, dirDemise], axis = 0).drop_duplicates().reset_index(drop = True)
dirDates

,Symbol,Date of Study
0,20MICRONS,1988-03-29
1,20MICRONS,1998-07-02
2,20MICRONS,2000-04-10
3,20MICRONS,2001-01-29
4,20MICRONS,2001-02-27
...,...,...
58009,ZYLOG,2013-06-24
58010,ZYLOG,2015-11-14
58011,ZYLOG,2016-06-30
58012,ZYLOG,2016-08-12


### Merging Dates within dataMerged

In [32]:
dirMerged = dataMerged2.merge(dirDates, left_on = ["Symbol", "AsOnDate"], right_on = ["Symbol", "Date of Study"], how = "left")

In [33]:
dirMerged

,CompanyName,ProwessCode,Symbol,AsOnDate,ACP,pct,RF,RMRF,MF,SMB,HML,Date of Study
0,20 Microns Ltd.,11.0,20MICRONS,2008-10-06,16.82,NaN,0.069713,-6.381449,-6.311735,-0.373052,-0.566450,NaT
1,20 Microns Ltd.,11.0,20MICRONS,2008-10-07,15.05,-0.105232,0.023232,-0.669144,-0.645911,-1.502487,0.184699,NaT
2,20 Microns Ltd.,11.0,20MICRONS,2008-10-08,13.25,-0.119601,0.023232,-3.533362,-3.510130,-1.780674,0.072932,NaT
3,20 Microns Ltd.,11.0,20MICRONS,2008-10-10,11.60,-0.124528,0.045520,-7.052324,-7.006804,0.217126,0.354629,NaT
4,20 Microns Ltd.,11.0,20MICRONS,2008-10-13,12.32,0.062069,0.066863,5.042738,5.109602,-2.437097,0.145853,NaT
...,...,...,...,...,...,...,...,...,...,...,...,...
13465088,Zylog Systems Ltd.,275793.0,ZYLOG,2024-03-21,0.35,0.000000,0.018215,1.441807,1.460021,0.572866,1.270303,NaT
13465089,Zylog Systems Ltd.,275793.0,ZYLOG,2024-03-22,0.35,0.000000,0.018215,0.593324,0.611538,0.752002,0.124527,NaT
13465090,Zylog Systems Ltd.,275793.0,ZYLOG,2024-03-26,0.35,0.000000,0.072878,-0.044514,0.028364,-1.091597,0.440605,NaT
13465091,Zylog Systems Ltd.,275793.0,ZYLOG,2024-03-27,0.35,0.000000,0.018215,0.326702,0.344916,-0.165072,-0.188505,NaT


# Organic Functions

In [34]:
# Just Trying
# FIT OLS, with R style formulas
# res = smf.ols("pct - RF ~ RMRF + SMB + HML", data = dataMerged).fit()

In [35]:
# res.summary()

In [36]:
dirMerged.CompanyName.nunique()

3721

## 120 Day Event Study

### 120 day window OLS

In [37]:
# pre-event 120 days
# ignoring companies with <130 data points in full.

def OLS120(frame):
    frame = frame.reset_index(drop = True)
    results = []
    if len(frame) > 151:
        
    #outputFrame = pd.DataFrame( columns = ["Date of Study", "const", "RMRF", "SMB", "HML", "OLS120_r_squared", "OLS120_adjusted_r_squared", "OLS120_f_p_value"])
        
        for date in range(len(frame["Date of Study"])) :
            if not pd.isnull(frame.iloc[date]["Date of Study"]):
                if date >= 151:
                    ols = frame.iloc[ date - 151 : date - 31]
                    exog_vars = ["RMRF"]
                    endog = ols.pct - ols.RF
                    exog = sm.add_constant(ols[exog_vars])
                    rols = sm.OLS(endog, exog)
                    res = rols.fit()
    
                    outputFrame1 = res.params.to_frame().T
                    outputFrame1["OLS120_r_squared"] = res.rsquared
                    outputFrame1["OLS120_adjusted_r_squared"] = res.rsquared_adj
                    outputFrame1["OLS120_f_p_value"] = res.f_pvalue
                    outputFrame1["Date of Study"] = frame.iloc[date]["Date of Study"]
    
                    if not outputFrame1.isnull().all().all():  # Ensure it's not all NaNs
                        results.append(outputFrame1)
    
        return pd.concat(results, ignore_index=True) if results else pd.DataFrame()

In [38]:
ols120Param = dirMerged.groupby(by = ["CompanyName"]).progress_apply(OLS120).reset_index()
ols120Param = ols120Param.rename({"const":"OLS120_intercept", "RMRF":"OLS120_RMRF"}, axis = 1).drop("level_1", axis = 1)

  0%|          | 0/3721 [00:00<?, ?it/s]

C:\Users\SHIVAM\anaconda3\Lib\site-packages\tqdm\std.py:805: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return getattr(df, df_function)(wrapper, **kwargs)


In [39]:
ols120Param

,CompanyName,OLS120_intercept,OLS120_RMRF,OLS120_r_squared,OLS120_adjusted_r_squared,OLS120_f_p_value,Date of Study
0,20 Microns Ltd.,-0.016130,0.003310,0.028601,0.020369,0.064814,2009-07-29
1,20 Microns Ltd.,-0.015122,0.003345,0.029881,0.021660,0.059023,2009-07-30
2,20 Microns Ltd.,-0.012672,0.002374,0.016580,0.008245,0.161037,2009-08-27
3,20 Microns Ltd.,-0.027709,0.010815,0.109751,0.102207,0.000219,2011-04-29
4,20 Microns Ltd.,-0.034611,0.011129,0.088050,0.080322,0.000998,2014-08-06
...,...,...,...,...,...,...,...
34579,Zylog Systems Ltd.,-0.034858,0.007051,0.027495,0.019253,0.070302,2015-08-14
34580,Zylog Systems Ltd.,-0.025075,0.002161,0.002523,-0.005930,0.585868,2016-06-30
34581,Zylog Systems Ltd.,-0.028892,0.001915,0.002487,-0.005967,0.588591,2016-08-12
34582,Zylog Systems Ltd.,-0.028364,0.006545,0.020030,0.011725,0.123099,2016-11-23


In [40]:
ols120 = dirMerged.merge(ols120Param, left_on = ["CompanyName", "AsOnDate"], right_on = ["CompanyName", "Date of Study"], how = "left").drop(["Date of Study_x", "Date of Study_y"], axis = 1)

In [41]:
ols120.to_pickle(rf"{output_folder_path}\ols120_2.pkl")

In [42]:
del ols120Param
del ols120

## 150 Day Event Study

### 150 day window OLS

In [43]:
# pre-event 150 days
# ignoring companies with <130 data points in full.

def OLS150(frame):
    frame = frame.reset_index(drop = True)
    results = []
    if len(frame) > 181:
    
        #outputFrame = pd.DataFrame( columns = ["Date of Study", "const", "RMRF", "SMB", "HML", "OLS150_r_squared", "OLS150_adjusted_r_squared", "OLS150_f_p_value"])
        
        for date in range(len(frame["Date of Study"])) :
            if not pd.isnull(frame.iloc[date]["Date of Study"]):
                if date >= 181:
                    ols = frame.iloc[ date - 181 : date - 31]
                    exog_vars = ["RMRF"]
                    endog = ols.pct - ols.RF
                    exog = sm.add_constant(ols[exog_vars])
                    rols = sm.OLS(endog, exog)
                    res = rols.fit()
    
                    outputFrame1 = res.params.to_frame().T
                    outputFrame1["OLS150_r_squared"] = res.rsquared
                    outputFrame1["OLS150_adjusted_r_squared"] = res.rsquared_adj
                    outputFrame1["OLS150_f_p_value"] = res.f_pvalue
                    outputFrame1["Date of Study"] = frame.iloc[date]["Date of Study"]
    
                    if not outputFrame1.isnull().all().all():  # Ensure it's not all NaNs
                        results.append(outputFrame1)
    
        return pd.concat(results, ignore_index=True) if results else pd.DataFrame()

In [44]:
ols150Param = dirMerged.groupby(by = ["CompanyName"]).progress_apply(OLS150).reset_index()
ols150Param = ols150Param.rename({"const":"OLS150_intercept", "RMRF":"OLS150_RMRF"}, axis = 1).drop("level_1", axis = 1)

  0%|          | 0/3721 [00:00<?, ?it/s]

C:\Users\SHIVAM\anaconda3\Lib\site-packages\tqdm\std.py:805: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return getattr(df, df_function)(wrapper, **kwargs)


In [45]:
ols150Param

,CompanyName,OLS150_intercept,OLS150_RMRF,OLS150_r_squared,OLS150_adjusted_r_squared,OLS150_f_p_value,Date of Study
0,20 Microns Ltd.,-0.018705,0.005061,0.064182,0.057859,0.001760,2009-07-29
1,20 Microns Ltd.,-0.017426,0.004621,0.055583,0.049202,0.003680,2009-07-30
2,20 Microns Ltd.,-0.016317,0.004135,0.048609,0.042181,0.006706,2009-08-27
3,20 Microns Ltd.,-0.026340,0.010345,0.100758,0.094682,0.000076,2011-04-29
4,20 Microns Ltd.,-0.034378,0.008381,0.050199,0.043781,0.005847,2014-08-06
...,...,...,...,...,...,...,...
34417,Zylog Systems Ltd.,-0.034571,0.010030,0.061262,0.054919,0.002261,2015-08-14
34418,Zylog Systems Ltd.,-0.027604,0.002431,0.002971,-0.003765,0.507632,2016-06-30
34419,Zylog Systems Ltd.,-0.025477,0.002946,0.004509,-0.002217,0.414249,2016-08-12
34420,Zylog Systems Ltd.,-0.029032,0.004966,0.013618,0.006953,0.154986,2016-11-23


In [46]:
ols150 = dirMerged.merge(ols150Param, left_on = ["CompanyName", "AsOnDate"], right_on = ["CompanyName", "Date of Study"], how = "left").drop(["Date of Study_x", "Date of Study_y"], axis = 1)

In [47]:
ols150

,CompanyName,ProwessCode,Symbol,AsOnDate,ACP,pct,RF,RMRF,MF,SMB,HML,OLS150_intercept,OLS150_RMRF,OLS150_r_squared,OLS150_adjusted_r_squared,OLS150_f_p_value
0,20 Microns Ltd.,11.0,20MICRONS,2008-10-06,16.82,NaN,0.069713,-6.381449,-6.311735,-0.373052,-0.566450,NaN,NaN,NaN,NaN,NaN
1,20 Microns Ltd.,11.0,20MICRONS,2008-10-07,15.05,-0.105232,0.023232,-0.669144,-0.645911,-1.502487,0.184699,NaN,NaN,NaN,NaN,NaN
2,20 Microns Ltd.,11.0,20MICRONS,2008-10-08,13.25,-0.119601,0.023232,-3.533362,-3.510130,-1.780674,0.072932,NaN,NaN,NaN,NaN,NaN
3,20 Microns Ltd.,11.0,20MICRONS,2008-10-10,11.60,-0.124528,0.045520,-7.052324,-7.006804,0.217126,0.354629,NaN,NaN,NaN,NaN,NaN
4,20 Microns Ltd.,11.0,20MICRONS,2008-10-13,12.32,0.062069,0.066863,5.042738,5.109602,-2.437097,0.145853,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13465088,Zylog Systems Ltd.,275793.0,ZYLOG,2024-03-21,0.35,0.000000,0.018215,1.441807,1.460021,0.572866,1.270303,NaN,NaN,NaN,NaN,NaN
13465089,Zylog Systems Ltd.,275793.0,ZYLOG,2024-03-22,0.35,0.000000,0.018215,0.593324,0.611538,0.752002,0.124527,NaN,NaN,NaN,NaN,NaN
13465090,Zylog Systems Ltd.,275793.0,ZYLOG,2024-03-26,0.35,0.000000,0.072878,-0.044514,0.028364,-1.091597,0.440605,NaN,NaN,NaN,NaN,NaN
13465091,Zylog Systems Ltd.,275793.0,ZYLOG,2024-03-27,0.35,0.000000,0.018215,0.326702,0.344916,-0.165072,-0.188505,NaN,NaN,NaN,NaN,NaN


In [48]:
ols150.to_pickle(rf"{output_folder_path}\ols150_2.pkl")

In [49]:
del ols150Param
del ols150

## 180 Day Event Study

### 180 day window OLS

In [50]:
# pre-event 180 days
# ignoring companies with <130 data points in full.

def OLS180(frame):
    frame = frame.reset_index(drop = True)
    results = []
    if len(frame) > 211:
    
        #outputFrame = pd.DataFrame( columns = ["Date of Study", "const", "RMRF", "SMB", "HML", "OLS180_r_squared", "OLS180_adjusted_r_squared", "OLS180_f_p_value"])
        
        for date in range(len(frame["Date of Study"])) :
            if not pd.isnull(frame.iloc[date]["Date of Study"]):
                if date >= 211:
                    ols = frame.iloc[ date - 211 : date - 31]
                    exog_vars = ["RMRF"]
                    endog = ols.pct - ols.RF
                    exog = sm.add_constant(ols[exog_vars])
                    rols = sm.OLS(endog, exog)
                    res = rols.fit()
    
                    outputFrame1 = res.params.to_frame().T
                    outputFrame1["OLS180_r_squared"] = res.rsquared
                    outputFrame1["OLS180_adjusted_r_squared"] = res.rsquared_adj
                    outputFrame1["OLS180_f_p_value"] = res.f_pvalue
                    outputFrame1["Date of Study"] = frame.iloc[date]["Date of Study"]
    
                    if not outputFrame1.isnull().all().all():  # Ensure it's not all NaNs
                        results.append(outputFrame1)
    
        return pd.concat(results, ignore_index=True) if results else pd.DataFrame()

In [51]:
ols180Param = dirMerged.groupby(by = ["CompanyName"]).progress_apply(OLS180).reset_index()
ols180Param = ols180Param.rename({"const":"OLS180_intercept", "RMRF":"OLS180_RMRF"}, axis = 1).drop("level_1", axis = 1)

  0%|          | 0/3721 [00:00<?, ?it/s]

C:\Users\SHIVAM\anaconda3\Lib\site-packages\tqdm\std.py:805: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return getattr(df, df_function)(wrapper, **kwargs)


In [52]:
ols180Param

,CompanyName,OLS180_intercept,OLS180_RMRF,OLS180_r_squared,OLS180_adjusted_r_squared,OLS180_f_p_value,Date of Study
0,20 Microns Ltd.,-0.019785,0.006153,0.095433,0.090351,2.449373e-05,2009-08-27
1,20 Microns Ltd.,-0.024022,0.010026,0.072412,0.067200,2.594653e-04,2011-04-29
2,20 Microns Ltd.,-0.034278,0.008417,0.042347,0.036967,5.581963e-03,2014-08-06
3,20 Microns Ltd.,-0.034440,0.006402,0.034724,0.029301,1.225873e-02,2015-02-05
4,20 Microns Ltd.,-0.024020,0.018310,0.186137,0.181564,1.480156e-09,2017-05-04
...,...,...,...,...,...,...,...
34110,Zylog Systems Ltd.,-0.034362,0.011860,0.071526,0.066310,2.840090e-04,2015-08-14
34111,Zylog Systems Ltd.,-0.028974,0.005380,0.020708,0.015206,5.395059e-02,2016-06-30
34112,Zylog Systems Ltd.,-0.027501,0.003164,0.004900,-0.000691,3.504521e-01,2016-08-12
34113,Zylog Systems Ltd.,-0.030786,0.004640,0.014079,0.008541,1.126326e-01,2016-11-23


In [53]:
ols180 = dirMerged.merge(ols180Param, left_on = ["CompanyName", "AsOnDate"], right_on = ["CompanyName", "Date of Study"], how = "left").drop(["Date of Study_x", "Date of Study_y"], axis = 1)

In [54]:
ols180

,CompanyName,ProwessCode,Symbol,AsOnDate,ACP,pct,RF,RMRF,MF,SMB,HML,OLS180_intercept,OLS180_RMRF,OLS180_r_squared,OLS180_adjusted_r_squared,OLS180_f_p_value
0,20 Microns Ltd.,11.0,20MICRONS,2008-10-06,16.82,NaN,0.069713,-6.381449,-6.311735,-0.373052,-0.566450,NaN,NaN,NaN,NaN,NaN
1,20 Microns Ltd.,11.0,20MICRONS,2008-10-07,15.05,-0.105232,0.023232,-0.669144,-0.645911,-1.502487,0.184699,NaN,NaN,NaN,NaN,NaN
2,20 Microns Ltd.,11.0,20MICRONS,2008-10-08,13.25,-0.119601,0.023232,-3.533362,-3.510130,-1.780674,0.072932,NaN,NaN,NaN,NaN,NaN
3,20 Microns Ltd.,11.0,20MICRONS,2008-10-10,11.60,-0.124528,0.045520,-7.052324,-7.006804,0.217126,0.354629,NaN,NaN,NaN,NaN,NaN
4,20 Microns Ltd.,11.0,20MICRONS,2008-10-13,12.32,0.062069,0.066863,5.042738,5.109602,-2.437097,0.145853,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13465088,Zylog Systems Ltd.,275793.0,ZYLOG,2024-03-21,0.35,0.000000,0.018215,1.441807,1.460021,0.572866,1.270303,NaN,NaN,NaN,NaN,NaN
13465089,Zylog Systems Ltd.,275793.0,ZYLOG,2024-03-22,0.35,0.000000,0.018215,0.593324,0.611538,0.752002,0.124527,NaN,NaN,NaN,NaN,NaN
13465090,Zylog Systems Ltd.,275793.0,ZYLOG,2024-03-26,0.35,0.000000,0.072878,-0.044514,0.028364,-1.091597,0.440605,NaN,NaN,NaN,NaN,NaN
13465091,Zylog Systems Ltd.,275793.0,ZYLOG,2024-03-27,0.35,0.000000,0.018215,0.326702,0.344916,-0.165072,-0.188505,NaN,NaN,NaN,NaN,NaN


In [55]:
ols180.to_pickle(rf"{output_folder_path}\ols180_2.pkl")

In [56]:
del ols180Param
del ols180

## 210 Day Event Study

### 210 day window OLS

In [57]:
# pre-event 210 days
# ignoring companies with <240 data points in full.

def OLS210(frame):
    frame = frame.reset_index(drop = True)
    results = []
    if len(frame) > 241:
    
        #outputFrame = pd.DataFrame( columns = ["Date of Study", "const", "RMRF", "SMB", "HML", "OLS210_r_squared", "OLS210_adjusted_r_squared", "OLS210_f_p_value"])
        
        for date in range(len(frame["Date of Study"])) :
            if not pd.isnull(frame.iloc[date]["Date of Study"]):
                if date >= 241:
                    ols = frame.iloc[ date - 241 : date - 31]
                    exog_vars = ["RMRF"]
                    endog = ols.pct - ols.RF
                    exog = sm.add_constant(ols[exog_vars])
                    rols = sm.OLS(endog, exog)
                    res = rols.fit()
    
                    outputFrame1 = res.params.to_frame().T
                    outputFrame1["OLS210_r_squared"] = res.rsquared
                    outputFrame1["OLS210_adjusted_r_squared"] = res.rsquared_adj
                    outputFrame1["OLS210_f_p_value"] = res.f_pvalue
                    outputFrame1["Date of Study"] = frame.iloc[date]["Date of Study"]
    
                    if not outputFrame1.isnull().all().all():  # Ensure it's not all NaNs
                        results.append(outputFrame1)
    
        return pd.concat(results, ignore_index=True) if results else pd.DataFrame()

In [58]:
ols210Param = dirMerged.groupby(by = ["CompanyName"]).progress_apply(OLS210).reset_index()
ols210Param = ols210Param.rename({"const":"OLS210_intercept", "RMRF":"OLS210_RMRF"}, axis = 1).drop("level_1", axis = 1)

  0%|          | 0/3721 [00:00<?, ?it/s]

C:\Users\SHIVAM\anaconda3\Lib\site-packages\tqdm\std.py:805: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return getattr(df, df_function)(wrapper, **kwargs)


In [59]:
ols210Param

,CompanyName,OLS210_intercept,OLS210_RMRF,OLS210_r_squared,OLS210_adjusted_r_squared,OLS210_f_p_value,Date of Study
0,20 Microns Ltd.,-0.023121,0.009385,0.073740,0.069287,6.696283e-05,2011-04-29
1,20 Microns Ltd.,-0.035117,0.005551,0.026592,0.021912,1.803598e-02,2014-08-06
2,20 Microns Ltd.,-0.034893,0.006952,0.038823,0.034202,4.151230e-03,2015-02-05
3,20 Microns Ltd.,-0.024740,0.017188,0.166374,0.162366,7.998804e-10,2017-05-04
4,20 Microns Ltd.,-0.025401,0.017863,0.233449,0.229763,1.106707e-13,2019-05-28
...,...,...,...,...,...,...,...
33920,Zylog Systems Ltd.,-0.035294,0.013010,0.077200,0.072764,4.444277e-05,2015-08-14
33921,Zylog Systems Ltd.,-0.027771,0.008411,0.038518,0.033895,4.304770e-03,2016-06-30
33922,Zylog Systems Ltd.,-0.028711,0.005695,0.022120,0.017419,3.120744e-02,2016-08-12
33923,Zylog Systems Ltd.,-0.027389,0.004182,0.010090,0.005331,1.468822e-01,2016-11-23


In [60]:
ols210 = dirMerged.merge(ols210Param, left_on = ["CompanyName", "AsOnDate"], right_on = ["CompanyName", "Date of Study"], how = "left").drop(["Date of Study_x", "Date of Study_y"], axis = 1)

In [61]:
ols210

,CompanyName,ProwessCode,Symbol,AsOnDate,ACP,pct,RF,RMRF,MF,SMB,HML,OLS210_intercept,OLS210_RMRF,OLS210_r_squared,OLS210_adjusted_r_squared,OLS210_f_p_value
0,20 Microns Ltd.,11.0,20MICRONS,2008-10-06,16.82,NaN,0.069713,-6.381449,-6.311735,-0.373052,-0.566450,NaN,NaN,NaN,NaN,NaN
1,20 Microns Ltd.,11.0,20MICRONS,2008-10-07,15.05,-0.105232,0.023232,-0.669144,-0.645911,-1.502487,0.184699,NaN,NaN,NaN,NaN,NaN
2,20 Microns Ltd.,11.0,20MICRONS,2008-10-08,13.25,-0.119601,0.023232,-3.533362,-3.510130,-1.780674,0.072932,NaN,NaN,NaN,NaN,NaN
3,20 Microns Ltd.,11.0,20MICRONS,2008-10-10,11.60,-0.124528,0.045520,-7.052324,-7.006804,0.217126,0.354629,NaN,NaN,NaN,NaN,NaN
4,20 Microns Ltd.,11.0,20MICRONS,2008-10-13,12.32,0.062069,0.066863,5.042738,5.109602,-2.437097,0.145853,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13465088,Zylog Systems Ltd.,275793.0,ZYLOG,2024-03-21,0.35,0.000000,0.018215,1.441807,1.460021,0.572866,1.270303,NaN,NaN,NaN,NaN,NaN
13465089,Zylog Systems Ltd.,275793.0,ZYLOG,2024-03-22,0.35,0.000000,0.018215,0.593324,0.611538,0.752002,0.124527,NaN,NaN,NaN,NaN,NaN
13465090,Zylog Systems Ltd.,275793.0,ZYLOG,2024-03-26,0.35,0.000000,0.072878,-0.044514,0.028364,-1.091597,0.440605,NaN,NaN,NaN,NaN,NaN
13465091,Zylog Systems Ltd.,275793.0,ZYLOG,2024-03-27,0.35,0.000000,0.018215,0.326702,0.344916,-0.165072,-0.188505,NaN,NaN,NaN,NaN,NaN


In [62]:
ols210.to_pickle(rf"{output_folder_path}\ols210_2.pkl")

In [63]:
ols210

,CompanyName,ProwessCode,Symbol,AsOnDate,ACP,pct,RF,RMRF,MF,SMB,HML,OLS210_intercept,OLS210_RMRF,OLS210_r_squared,OLS210_adjusted_r_squared,OLS210_f_p_value
0,20 Microns Ltd.,11.0,20MICRONS,2008-10-06,16.82,NaN,0.069713,-6.381449,-6.311735,-0.373052,-0.566450,NaN,NaN,NaN,NaN,NaN
1,20 Microns Ltd.,11.0,20MICRONS,2008-10-07,15.05,-0.105232,0.023232,-0.669144,-0.645911,-1.502487,0.184699,NaN,NaN,NaN,NaN,NaN
2,20 Microns Ltd.,11.0,20MICRONS,2008-10-08,13.25,-0.119601,0.023232,-3.533362,-3.510130,-1.780674,0.072932,NaN,NaN,NaN,NaN,NaN
3,20 Microns Ltd.,11.0,20MICRONS,2008-10-10,11.60,-0.124528,0.045520,-7.052324,-7.006804,0.217126,0.354629,NaN,NaN,NaN,NaN,NaN
4,20 Microns Ltd.,11.0,20MICRONS,2008-10-13,12.32,0.062069,0.066863,5.042738,5.109602,-2.437097,0.145853,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13465088,Zylog Systems Ltd.,275793.0,ZYLOG,2024-03-21,0.35,0.000000,0.018215,1.441807,1.460021,0.572866,1.270303,NaN,NaN,NaN,NaN,NaN
13465089,Zylog Systems Ltd.,275793.0,ZYLOG,2024-03-22,0.35,0.000000,0.018215,0.593324,0.611538,0.752002,0.124527,NaN,NaN,NaN,NaN,NaN
13465090,Zylog Systems Ltd.,275793.0,ZYLOG,2024-03-26,0.35,0.000000,0.072878,-0.044514,0.028364,-1.091597,0.440605,NaN,NaN,NaN,NaN,NaN
13465091,Zylog Systems Ltd.,275793.0,ZYLOG,2024-03-27,0.35,0.000000,0.018215,0.326702,0.344916,-0.165072,-0.188505,NaN,NaN,NaN,NaN,NaN
